<a href="https://colab.research.google.com/github/aditya01-dev/Natural-Language-Processing-Lab/blob/main/NLP_Practical_30_Transformer_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Name: Aditya kumar Singh
import sys
# Practical 30: Transformer-Based Chatbot
# Objective: Build a chatbot using a pretrained transformer language model.

# 1. Install required library
!pip -q install transformers

# 2. Import required libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 3. Load pretrained conversational model
print("Loading model and tokenizer...")
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model loaded successfully on {device}!")

Loading model and tokenizer...


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Model loaded successfully on cpu!


In [ ]:
# 4. Clean & Modular Chatbot Engine
chat_history_ids = None

def generate_response(user_message):
    global chat_history_ids

    # Encode the new user input and append the eos_token
    new_user_input_ids = tokenizer.encode(user_message + tokenizer.eos_token, return_tensors='pt').to(device)

    # Append the new user input tokens to the chat history
    bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if chat_history_ids is not None else new_user_input_ids

    # Generate a response from the model
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.75
    )

    # Pretty-print only the newly generated tokens from the model
    decoded_response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    return decoded_response

def reset_chat():
    global chat_history_ids
    chat_history_ids = None
    print("Chat history reset successfully!")

# Quick test
user_msg = "Hello! Who are you?"
reply = generate_response(user_msg)
print(f"User: {user_msg}")
print(f"Bot: {reply}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


User: Hello! Who are you?
Bot: Hi there !


In [ ]:
# 5. Interactive Chat UI using Colab Widgets
import ipywidgets as widgets
from IPython.display import display, clear_output

# Suppression of tokenization space warnings
tokenizer.clean_up_tokenization_spaces = False

# Create interactive widget elements
text_input = widgets.Text(
    value='',
    placeholder='Type your message here...',
    description='You:',
    disabled=False,
    layout=widgets.Layout(width='70%')
)

send_button = widgets.Button(
    description='Send',
    button_style='success',
    tooltip='Send message to chatbot',
    icon='paper-plane'
)

reset_button = widgets.Button(
    description='Reset History',
    button_style='danger',
    tooltip='Clear conversation history',
    icon='trash'
)

output_area = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='10px 0'))

# UI interactions
def on_send_clicked(b):
    user_text = text_input.value.strip()
    if not user_text:
        return

    with output_area:
        print(f"You: {user_text}")
        text_input.value = '' # Clear the input box immediately

        # Generate response
        try:
            bot_reply = generate_response(user_text)
            print(f"Chatbot: {bot_reply}\n" + "-"*40)
        except Exception as e:
            print(f"System Error: {str(e)}")

def on_reset_clicked(b):
    reset_chat()
    with output_area:
        clear_output()
        print("Conversation cleared. Start a new topic!")

send_button.on_click(on_send_clicked)
text_input.on_submit(on_send_clicked)
reset_button.on_click(on_reset_clicked)

# Layout setup
input_box = widgets.HBox([text_input, send_button, reset_button])
ui_container = widgets.VBox([output_area, input_box])

# Display the styled UI
print("=== Interactive Chat Window ===")
display(ui_container)

=== Interactive Chat Window ===


Chat history reset successfully!
Chat history reset successfully!
